# Experiment: Feature Signal Combo Lab

Objective:
- Sweep feature, indicator, signal, and target parameters from a tracked experiment config.
- Inspect which feature and signal combinations align best with the final trading signal and OOS equity profile.

Runtime note:
- Run this inside the dev container / Docker environment so Plotly, Streamlit, XGBoost, and the repo dependencies match the tracked setup.
- Start with `shock_meta_long_only`; if you switch experiment family, update the combo column selections after re-running the feature catalog cell.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / ".git").exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from notebooks.experiment_lab_support import (
    build_analysis_frame_from_result,
    build_feature_signal_catalog,
    build_feature_signal_combo_frame,
    build_feature_signal_combo_frames,
    build_mutated_config,
    build_summary_frame,
    get_experiment_spec,
    load_logged_artifact_bundle,
    merge_nested_overrides,
    plot_equity_drawdown,
    plot_feature_signal_combo,
    plot_feature_signal_combo_suite,
    plot_positions_turnover,
    plot_price_signal_probability,
    plot_returns_distribution,
    plotly_chart_config,
    run_experiment_from_config,
)

pd.options.display.float_format = "{:,.6f}".format

EXPERIMENT_KEY = "shock_meta_long_only"
EXPERIMENT = get_experiment_spec(EXPERIMENT_KEY)


def show_figures(figures: dict[str, object]) -> None:
    for name, fig in figures.items():
        display(Markdown(f"### {name}"))
        fig.show(config=plotly_chart_config())


EXPERIMENT


In [ ]:
logged_frame, logged_summary, logged_config = load_logged_artifact_bundle(EXPERIMENT_KEY)

display(build_summary_frame(logged_summary))
pd.Series(
    {
        "config_path": EXPERIMENT["config_path"],
        "logged_run_dir": EXPERIMENT["logged_run_dir"],
        "model_feature_count": len(logged_config.get("model", {}).get("feature_cols", []) or []),
        "signal_col": logged_config.get("backtest", {}).get("signal_col"),
    }
).to_frame("value")


## Plan

- Sweep blocks:
  - named feature steps via `FEATURE_STEP_UPDATES`
  - signal thresholds via `SIGNAL_PARAM_UPDATES`
  - model hyperparameters and feature subset via `MODEL_PARAM_UPDATES` and `MODEL_FEATURE_COLS`
  - target, risk, and backtest settings via the dedicated update dicts
- Workflow:
  - keep the experiment family fixed
  - rerun the config
  - inspect the generated feature catalog
  - compare single combos and a multi-combo suite


In [ ]:
# Edit these blocks before re-running the notebook.
FEATURE_STEP_UPDATES = {
    "shock_context": {
        "ret_z_threshold": 2.5,
        "atr_mult_threshold": 1.75,
        "post_shock_active_bars": 2,
    },
    "regime_context": {
        "vol_short_window": 24,
        "vol_long_window": 168,
        "trend_fast_span": 24,
        "trend_slow_span": 72,
        "vol_ratio_high_threshold": 1.35,
        "vol_ratio_low_threshold": 0.80,
    },
    "bollinger": {
        "window": 24,
        "n_std": 2.0,
    },
    "rsi": {
        "windows": [2, 14],
    },
    "lags": {
        "lags": [1],
    },
}

SIGNAL_PARAM_UPDATES = {
    "upper": 0.555,
    "upper_exit": 0.49,
    "lower": 0.43,
    "lower_exit": 0.48,
    "mode": "long_only",
}

MODEL_PARAM_UPDATES = {
    "max_depth": 4,
    "n_estimators": 350,
    "learning_rate": 0.03,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
}

TARGET_UPDATES = {
    "max_holding": 24,
    "upper_mult": 1.5,
    "lower_mult": 1.5,
}

RISK_UPDATES = {
    "dd_guard": {
        "cooloff_bars": 48,
        "max_drawdown": 0.12,
        "rearm_drawdown": 0.08,
    }
}

BACKTEST_UPDATES = {}
EXTRA_OVERRIDES = {}

# Set to None to keep the tracked feature set, or pass a reduced / expanded list.
MODEL_FEATURE_COLS = None

{
    "feature_steps": list(FEATURE_STEP_UPDATES),
    "model_feature_cols": MODEL_FEATURE_COLS,
    "signal_updates": SIGNAL_PARAM_UPDATES,
}


In [ ]:
cfg = build_mutated_config(
    EXPERIMENT["config_path"],
    overrides=EXTRA_OVERRIDES,
    feature_step_updates=FEATURE_STEP_UPDATES,
    signal_param_updates=SIGNAL_PARAM_UPDATES,
    model_param_updates=MODEL_PARAM_UPDATES,
    model_feature_cols=MODEL_FEATURE_COLS,
    target_updates=TARGET_UPDATES,
    risk_updates=RISK_UPDATES,
    backtest_updates=BACKTEST_UPDATES,
)

feature_steps = pd.DataFrame(
    [
        {
            "step": step.get("step"),
            "enabled": step.get("enabled", True),
            "params": step.get("params", {}),
        }
        for step in cfg.get("features", [])
    ]
)

display(feature_steps)
pd.Series(
    {
        "signal_col": cfg.get("backtest", {}).get("signal_col"),
        "model_feature_count": len(cfg.get("model", {}).get("feature_cols", []) or []),
        "target_kind": cfg.get("model", {}).get("target", {}).get("kind"),
        "cooloff_bars": cfg.get("risk", {}).get("dd_guard", {}).get("cooloff_bars"),
    }
).to_frame("value")


In [ ]:
result = run_experiment_from_config(
    cfg,
    config_path=EXPERIMENT["config_path"],
    logging_enabled=False,
)
frame = build_analysis_frame_from_result(result)
summary = dict(result.evaluation.get("primary_summary", {}) or {})
catalog = build_feature_signal_catalog(frame, result.config)
signal_col = str(result.config["backtest"]["signal_col"])

build_summary_frame(summary)


In [ ]:
display(pd.Series(catalog["model_feature_cols"], name="model_feature_cols").to_frame())
display(pd.Series(catalog["feature_candidates"], name="feature_candidates").to_frame().head(80))
display(pd.Series(catalog["signal_candidates"], name="signal_candidates").to_frame())
display(pd.Series(catalog["prediction_candidates"], name="prediction_candidates").to_frame())


In [ ]:
FEATURE_COLS = [
    "shock_strength",
    "shock_distance_ema",
    "regime_vol_ratio_24_168",
]
SIGNAL_COLS = [
    "shock_side_contrarian_active",
    signal_col,
    "pred_prob",
]

FEATURE_WEIGHTS = {
    "shock_strength": 1.2,
    "shock_distance_ema": 1.0,
    "regime_vol_ratio_24_168": 0.8,
}
SIGNAL_WEIGHTS = {
    "shock_side_contrarian_active": 0.8,
    signal_col: 1.0,
    "pred_prob": 1.2,
}

combo_frame = build_feature_signal_combo_frame(
    frame,
    feature_cols=FEATURE_COLS,
    signal_cols=SIGNAL_COLS,
    feature_weights=FEATURE_WEIGHTS,
    signal_weights=SIGNAL_WEIGHTS,
    normalize=True,
)

combo_frame[
    [
        *(f"{column}__scaled" for column in FEATURE_COLS),
        *(f"{column}__scaled" for column in SIGNAL_COLS),
        "feature_combo",
        "signal_combo",
        "joint_combo",
    ]
].tail(20)


In [ ]:
plot_feature_signal_combo(
    frame,
    title="Feature / Signal Combo Overlay",
    feature_cols=FEATURE_COLS,
    signal_cols=SIGNAL_COLS,
    feature_weights=FEATURE_WEIGHTS,
    signal_weights=SIGNAL_WEIGHTS,
    normalize=True,
)


In [ ]:
TRIAL_COLUMNS = [
    "trial",
    "sharpe",
    "sortino",
    "cumulative_return",
    "net_pnl",
    "max_drawdown",
    "avg_turnover",
]


def run_trial_grid(trial_specs: dict[str, dict[str, object]]) -> tuple[pd.DataFrame, dict[str, dict[str, object]]]:
    rows = []
    trial_outputs: dict[str, dict[str, object]] = {}

    for trial_name, trial_spec in trial_specs.items():
        trial_cfg = build_mutated_config(
            EXPERIMENT["config_path"],
            overrides=merge_nested_overrides(EXTRA_OVERRIDES, trial_spec.get("overrides", {}) or {}),
            feature_step_updates=merge_nested_overrides(
                FEATURE_STEP_UPDATES,
                trial_spec.get("feature_step_updates", {}) or {},
            ),
            signal_param_updates=merge_nested_overrides(
                SIGNAL_PARAM_UPDATES,
                trial_spec.get("signal_param_updates", {}) or {},
            ),
            model_param_updates=merge_nested_overrides(
                MODEL_PARAM_UPDATES,
                trial_spec.get("model_param_updates", {}) or {},
            ),
            model_feature_cols=trial_spec.get("model_feature_cols", MODEL_FEATURE_COLS),
            target_updates=merge_nested_overrides(
                TARGET_UPDATES,
                trial_spec.get("target_updates", {}) or {},
            ),
            risk_updates=merge_nested_overrides(
                RISK_UPDATES,
                trial_spec.get("risk_updates", {}) or {},
            ),
            backtest_updates=merge_nested_overrides(
                BACKTEST_UPDATES,
                trial_spec.get("backtest_updates", {}) or {},
            ),
        )
        trial_result = run_experiment_from_config(
            trial_cfg,
            config_path=EXPERIMENT["config_path"],
            logging_enabled=False,
        )
        trial_summary = dict(trial_result.evaluation.get("primary_summary", {}) or {})
        trial_frame = build_analysis_frame_from_result(trial_result)

        rows.append(
            {
                "trial": trial_name,
                "sharpe": trial_summary.get("sharpe"),
                "sortino": trial_summary.get("sortino"),
                "cumulative_return": trial_summary.get("cumulative_return"),
                "net_pnl": trial_summary.get("net_pnl"),
                "max_drawdown": trial_summary.get("max_drawdown"),
                "avg_turnover": trial_summary.get("avg_turnover"),
            }
        )
        trial_outputs[trial_name] = {
            "config": trial_cfg,
            "result": trial_result,
            "frame": trial_frame,
            "summary": trial_summary,
        }

    if not rows:
        return pd.DataFrame(columns=TRIAL_COLUMNS), trial_outputs

    trial_table = pd.DataFrame(rows).sort_values("sharpe", ascending=False).reset_index(drop=True)
    return trial_table[TRIAL_COLUMNS], trial_outputs


TRIAL_SPECS = {
    # "higher_shock_threshold": {
    #     "feature_step_updates": {
    #         "shock_context": {
    #             "ret_z_threshold": 2.8,
    #             "atr_mult_threshold": 2.0,
    #         }
    #     }
    # },
    # "tighter_probability_gate": {
    #     "signal_param_updates": {
    #         "upper": 0.60,
    #         "upper_exit": 0.52,
    #     }
    # },
}

TRIAL_SPECS


In [ ]:
trial_table, trial_outputs = run_trial_grid(TRIAL_SPECS)
trial_table


In [ ]:
COMBO_SPECS = [
    {
        "name": "shock_regime_filter",
        "title": "Shock + Regime Filter",
        "feature_cols": [
            "shock_strength",
            "shock_ret_z_1h",
            "regime_vol_ratio_24_168",
        ],
        "signal_cols": [
            "shock_side_contrarian_active",
            signal_col,
        ],
        "feature_weights": {
            "shock_strength": 1.2,
            "shock_ret_z_1h": 1.0,
            "regime_vol_ratio_24_168": 0.8,
        },
    },
    {
        "name": "mean_reversion_stack",
        "title": "Bollinger + RSI Stack",
        "feature_cols": [
            "bb_percent_b_24_2.0",
            "close_rsi_2",
            "close_rsi_14",
        ],
        "signal_cols": [
            signal_col,
            "pred_prob",
        ],
        "feature_weights": {
            "bb_percent_b_24_2.0": 1.1,
            "close_rsi_2": 1.0,
            "close_rsi_14": 0.8,
        },
    },
    {
        "name": "volatility_impulse_stack",
        "title": "ATR + Lagged Return Impulse",
        "feature_cols": [
            "shock_atr_multiple_1h",
            "shock_atr_multiple_4h",
            "lag_close_logret_1",
        ],
        "signal_cols": [
            "pred_prob",
            signal_col,
        ],
        "feature_weights": {
            "shock_atr_multiple_1h": 1.0,
            "shock_atr_multiple_4h": 1.0,
            "lag_close_logret_1": 0.6,
        },
    },
]

combo_frames = build_feature_signal_combo_frames(frame, COMBO_SPECS)
display(combo_frames["shock_regime_filter"].tail(10))

combo_figures = plot_feature_signal_combo_suite(frame, COMBO_SPECS)
show_figures(combo_figures)


In [ ]:
show_figures(
    {
        "price_signal_probability": plot_price_signal_probability(
            frame,
            title=f"{EXPERIMENT['label']}: Price / Signal / Probability",
            signal_col=signal_col,
        ),
        "equity_drawdown": plot_equity_drawdown(
            frame,
            title=f"{EXPERIMENT['label']}: Equity / Drawdown",
        ),
        "positions_turnover": plot_positions_turnover(
            frame,
            title=f"{EXPERIMENT['label']}: Position / Turnover / Costs",
        ),
        "returns_distribution": plot_returns_distribution(
            frame,
            title=f"{EXPERIMENT['label']}: Net Return Distribution",
        ),
    }
)


## Next steps

- Promote only parameter sets that improve OOS Sharpe or drawdown without exploding turnover.
- If you change the experiment family, rerun the catalog cell before editing the combo specs.
- Use the Streamlit explorer for longer manual browsing; Plotly zoom and pan work in both notebook and Streamlit.
